In [1]:
!pip install flask-ngrok
!pip install pyngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import google.generativeai as genai
import re

# Initialize the Flask app
app = Flask(__name__)

# Set up ngrok for public URL access
NGROK_AUTH_TOKEN = "YOUR_NGROK_API_KEY"  # Replace with your actual Ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000).public_url

# Configure the Gemini API
GENAI_API_KEY = "YOUR_GEMINI_API_KEY"  # Replace with your actual Gemini API key
genai.configure(api_key=GENAI_API_KEY)

conversation_history = []  # Store conversation history

# 🔹 Function to fetch response from Gemini AI
def get_gemini_response(query):
    """Fetch response from Gemini AI based on the job-related query and save the response."""
    prompt = f"""
      You are a professional career advisor. Respond only to queries related to careers, jobs, courses, or skills.

      **Query:** {query}

      - If the query is related to careers, jobs, courses, or skills, provide a detailed, structured, and professional response.
      - If the query is unrelated, politely decline to answer.
      - Do **not** start with "Okay, I understand" or similar phrases. Make your answer straight to the point directly.

      Ensure the response is professional, informative, and **concise (maximum 10 lines).**
      """

    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(prompt)
        response_text = response.text.strip() if response else "No response available."

        # Save response to a file
        with open("responses.txt", "a", encoding="utf-8") as file:
            file.write(f"Query: {query}\nResponse: {response_text}\n{'-'*50}\n")

        return response_text
    except Exception as e:
        print(f"Error generating response: {e}")
        return f"Error processing request: {str(e)}"

# 🔹 Function to find the top 3 matching advisors
def match_advisors(user_data, advisors):
    """Find the best matching career advisors based on the user's skills and needs."""
    user_skills = ", ".join(user_data.get("skills", []))  # Convert skills to string

    advisor_info = "\n".join([
        f"ID: {a['id']}, Name: {a['name']}, Specialization: {a['specialization']}"
        for a in advisors
    ])

    prompt = f"""
      You are an AI system designed to match users with the most relevant career advisors.

      **User Details:**
      - User ID: {user_data.get("user_id")}
      - Skills: {user_skills}

      **Available Advisors:**
      {advisor_info}

      Select up to 3 advisors who are the best match for this user based on their skills.
      Provide only the advisor IDs in a **comma-separated format** (e.g., "1, 3, 5").
    """

    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(prompt)
        response_text = response.text.strip() if response else ""

        # Extract advisor IDs using regex
        matches = re.findall(r"\d+", response_text)
        selected_advisors = [a for a in advisors if str(a["id"]) in matches]

        return selected_advisors[:3]  # Return top 3 advisors
    except Exception as e:
        print(f"Error selecting advisors: {e}")
        return []

# 🔹 Route 1: `/job_info` - Get job-related information
@app.route("/job_info", methods=["POST"])
def job_info():
    """API endpoint to get job information from Gemini AI."""
    data = request.json
    query = data.get("query")

    if not query:
        return jsonify({"error": "Please provide a job-related query."}), 400

    response = get_gemini_response(query)

    conversation_history.append(("User", query))
    conversation_history.append(("Gemini", response))

    return jsonify({
        "query": query,
        "response": response
    }), 200

# 🔹 Route 2: `/match_advisors` - Find the best 1-3 career advisors
@app.route("/match_advisors", methods=["POST"])
def match_advisor_route():
    """API endpoint to match a user with the best career advisors."""
    data = request.json
    user_data = data.get("user", {})
    advisors = data.get("advisors", [])

    if not user_data or not advisors:
        return jsonify({"error": "Please provide user data and advisor list."}), 400

    best_advisors = match_advisors(user_data, advisors)

    return jsonify({
        "user_id": user_data.get("user_id"),
        "matched_advisors": best_advisors if best_advisors else "No suitable advisors found"
    }), 200

if __name__ == "__main__":
    print(f"App running at {public_url}")
    app.run(port=5000)


App running at https://afda-34-86-6-133.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 12:08:52] "POST /job_info HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2025 12:08:59] "POST /match_advisors HTTP/1.1" 200 -
